In [ ]:
# Setup: repo path, paper folder, and fixed domain list.
import sys
from pathlib import Path

repo_root = Path.cwd().resolve()
if repo_root.name == "notebooks":
    repo_root = repo_root.parent
sys.path.append(str(repo_root))

papers_dir = (repo_root / "data" / "papers" / "json" / "all_papers").resolve()
DOMAIN_NAMES = [f"domain_{i}" for i in range(1, 8)]

print(f"Papers dir: {papers_dir}")
print(f"Domains: {', '.join(DOMAIN_NAMES)}")


Papers dir: /home/user/Thesis/conservation-llm-critical-appraisal/data/papers/json/1_to_100
Domains: domain_1, domain_2, domain_3, domain_4, domain_5, domain_6, domain_7


In [4]:
# Imports.
import json
from datetime import datetime, timezone

from domain_shot import build_question_scoring_prompt as qs
from domain_shot import build_tree_guided_prompt as tq
from domain_shot.evaluation import DEFAULT_MODEL, call_llm_and_process

In [5]:
# Load shared text and paper list (numeric filename order).
intro_text = tq.load_default_guidance_intro()

domain_questions_by_domain = {}
decision_trees_by_domain = {}
for domain_name in DOMAIN_NAMES:
    domain_questions_by_domain[domain_name] = tq.load_domain_questions(domain_name)
    decision_trees_by_domain[domain_name] = tq.load_domain_decision_tree(domain_name)

paper_files = sorted(
    [p for p in papers_dir.glob("*.json")],
    key=lambda p: int(p.stem) if p.stem.isdigit() else p.stem,
 )

print(f"Papers found: {len(paper_files)}")


Papers found: 97


In [6]:
import subprocess

#run cost check
result = subprocess.run(
    ["python", str(repo_root / "tests" / "api_credit_test.py")],
    capture_output=True,
    text=True
)
print(result.stdout)
print(result.stderr)

{
  "data": {
    "label": "sk-or-v1-012...a08",
    "is_management_key": false,
    "is_provisioning_key": false,
    "limit": 1000,
    "limit_reset": null,
    "limit_remaining": 391.199121839,
    "include_byok_in_limit": false,
    "usage": 608.800878161,
    "usage_daily": 165.411875,
    "usage_weekly": 415.33033,
    "usage_monthly": 569.390131772,
    "byok_usage": 0,
    "byok_usage_daily": 0,
    "byok_usage_weekly": 0,
    "byok_usage_monthly": 0,
    "is_free_tier": false,
    "expires_at": "2027-01-12T14:31:22.095Z",
    "creator_user_id": "user_35TUVIRsSsltNjDcienaMimoj8K",
    "rate_limit": {
      "requests": -1,
      "interval": "10s",
      "note": "This field is deprecated and safe to ignore."
    }
  }
}




In [7]:
# Run tree-guided prompt for all papers and all domains.
tree_guided_results = {}

for paper_file in paper_files:
    with open(paper_file, "r", encoding="utf-8") as f:
        paper_text = json.dumps(json.load(f), indent=2, ensure_ascii=False)
    domain_results = {}
    for domain_name in DOMAIN_NAMES:
        try:
            messages = tq.build_prompt_messages(
                domain_name=domain_name,
                intro_text=intro_text,
                decision_tree_text=decision_trees_by_domain[domain_name],
                domain_questions_text=domain_questions_by_domain[domain_name],
                paper_text=paper_text,
            )
            domain_results[domain_name] = call_llm_and_process(domain_name, messages, DEFAULT_MODEL)
        except Exception as exc:
            domain_results[domain_name] = {"error": str(exc)}
    tree_guided_results[paper_file.name] = domain_results

print(f"Tree-guided complete for {len(tree_guided_results)} papers")


Tree-guided complete for 97 papers


In [8]:
import subprocess

#run cost check
result = subprocess.run(
    ["python", str(repo_root / "tests" / "api_credit_test.py")],
    capture_output=True,
    text=True
)
print(result.stdout)
print(result.stderr)

{
  "data": {
    "label": "sk-or-v1-012...a08",
    "is_management_key": false,
    "is_provisioning_key": false,
    "limit": 1000,
    "limit_reset": null,
    "limit_remaining": 384.724636079,
    "include_byok_in_limit": false,
    "usage": 615.275363921,
    "usage_daily": 2.194572116,
    "usage_weekly": 421.80481576,
    "usage_monthly": 575.864617532,
    "byok_usage": 0,
    "byok_usage_daily": 0,
    "byok_usage_weekly": 0,
    "byok_usage_monthly": 0,
    "is_free_tier": false,
    "expires_at": "2027-01-12T14:31:22.095Z",
    "creator_user_id": "user_35TUVIRsSsltNjDcienaMimoj8K",
    "rate_limit": {
      "requests": -1,
      "interval": "10s",
      "note": "This field is deprecated and safe to ignore."
    }
  }
}




In [9]:
# Run question-scoring prompt for all papers and all domains.
question_scoring_results = {}

for paper_file in paper_files:
    with open(paper_file, "r", encoding="utf-8") as f:
        paper_text = json.dumps(json.load(f), indent=2, ensure_ascii=False)
    domain_results = {}
    for domain_name in DOMAIN_NAMES:
        try:
            messages = qs.build_prompt_messages(
                domain_name=domain_name,
                intro_text=intro_text,
                domain_questions_text=domain_questions_by_domain[domain_name],
                paper_text=paper_text,
            )
            domain_results[domain_name] = call_llm_and_process(domain_name, messages, DEFAULT_MODEL)
        except Exception as exc:
            domain_results[domain_name] = {"error": str(exc)}
    question_scoring_results[paper_file.name] = domain_results

print(f"Question-scoring complete for {len(question_scoring_results)} papers")


Question-scoring complete for 97 papers


In [12]:
import subprocess

#run cost check
result = subprocess.run(
    ["python", str(repo_root / "tests" / "api_credit_test.py")],
    capture_output=True,
    text=True
)
print(result.stdout)
print(result.stderr)

{
  "data": {
    "label": "sk-or-v1-012...a08",
    "is_management_key": false,
    "is_provisioning_key": false,
    "limit": 1000,
    "limit_reset": null,
    "limit_remaining": 343.717546716,
    "include_byok_in_limit": false,
    "usage": 656.282453284,
    "usage_daily": 24.210757785,
    "usage_weekly": 462.811905123,
    "usage_monthly": 616.871706895,
    "byok_usage": 0,
    "byok_usage_daily": 0,
    "byok_usage_weekly": 0,
    "byok_usage_monthly": 0,
    "is_free_tier": false,
    "expires_at": "2027-01-12T14:31:22.095Z",
    "creator_user_id": "user_35TUVIRsSsltNjDcienaMimoj8K",
    "rate_limit": {
      "requests": -1,
      "interval": "10s",
      "note": "This field is deprecated and safe to ignore."
    }
  }
}




In [11]:
# Save two separate output files (overwrite), with model suffix in filename.
timestamp_utc = datetime.now(timezone.utc).isoformat()
output_dir = repo_root / "output" / "all_papers_all_domains_two_prompts" / "1_to_100"
output_dir.mkdir(parents=True, exist_ok=True)

model_suffix = DEFAULT_MODEL.replace("/", "_").replace(":", "_").replace(" ", "_")
tree_output_path = output_dir / f"tree_guided_all_papers_{model_suffix}.json"
question_output_path = output_dir / f"question_scoring_all_papers_{model_suffix}.json"

tree_payload = {
    "model": DEFAULT_MODEL,
    "timestamp_utc": timestamp_utc,
    "domains": DOMAIN_NAMES,
    "results": tree_guided_results,
}
question_payload = {
    "model": DEFAULT_MODEL,
    "timestamp_utc": timestamp_utc,
    "domains": DOMAIN_NAMES,
    "results": question_scoring_results,
}

with open(tree_output_path, "w", encoding="utf-8") as f:
    json.dump(tree_payload, f, indent=2, ensure_ascii=False)
with open(question_output_path, "w", encoding="utf-8") as f:
    json.dump(question_payload, f, indent=2, ensure_ascii=False)

print(f"Saved: {tree_output_path}")
print(f"Saved: {question_output_path}")

Saved: /home/user/Thesis/conservation-llm-critical-appraisal/output/all_papers_all_domains_two_prompts/1_to_100/tree_guided_all_papers_z-ai_glm-5.2.json
Saved: /home/user/Thesis/conservation-llm-critical-appraisal/output/all_papers_all_domains_two_prompts/1_to_100/question_scoring_all_papers_z-ai_glm-5.2.json
